<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>

## Decision Variables, Objective, and Constraints · Continuing the Classroom Model

**The classroom system model is already fixed; the task now is to choose a cooling decision that satisfies every requirement and gives the lowest score.**

Lecture 01-1 supplied the state equation and evaluation chain. This lecture keeps them unchanged and makes the decision variables, objective, and constraints explicit.

> **01-1: frame the problem → build the system equation → simulate and evaluate**  
> **01-2: name the decision variables → write the objective → state the constraints → select a feasible decision**

The word *model* can refer to two related objects. We will name each one precisely:

| Object | What it contains | Job |
|:---|:---|:---|
| **System model** | State equation $F$ | Predict the temperature path produced by a cooling decision |
| **Optimization formulation** | Decision variables, objective, constraints, and the system model | Define which decision should be selected |

We do not build another physical model in 01-2. We reuse the classroom system and make its decision problem explicit.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="../01-1_systems_thinking/assets/05_system_optimization_wide_labeled.png" alt="Classroom optimization chain from cooling decision through temperature and raw performance to score and preferred candidate" width="720" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

### 1 · Carry forward the roles from 01-1

| Role | Classroom quantity | Treatment |
|:---|:---|:---|
| Fixed parameter | $a=0.12,\ b=0.012,\ c=0.45$ | Held fixed |
| External input | $T_t^{\mathrm{out}}=31\,^\circ\mathrm{C},\ N_t=20$ | Held fixed across candidates |
| System state | Indoor temperature $T_t$ | Produced by simulation |
| Decision variable | $u=(u_{\mathrm{early}},u_{\mathrm{late}})$, expanded into $u_t$ | Chosen directly |
| Performance output | Discomfort $D$, energy $E$ | Calculated after simulation |
| Constraint | Cooling, temperature, and energy limits | Determines feasibility |
| Hyperparameter | Energy weight $\lambda_E$ | Sets how $D$ and $E$ are compared |

These roles do not change between lectures. The new work is to express their relationships precisely enough to identify the preferred feasible decision.


### 2 · Name the decision without rebuilding the system

Lecture 01-1 defined a horizon of \(n=12\) decision steps and used one cooling level for the first six steps and another for the last six. The two directly chosen values form the **decision vector**

> $\displaystyle u=(u_{\mathrm{early}},u_{\mathrm{late}}).$

The vector expands into the 12-step cooling schedule

> $\displaystyle
u_t=
\begin{cases}
u_{\mathrm{early}}, & t=0,\ldots,5,\\
u_{\mathrm{late}}, & t=6,\ldots,11.
\end{cases}$

With \(n=12\), the state equation from 01-1 remains unchanged for \(t=0,\ldots,n-1\):

> $\displaystyle T_{t+1}=T_t+a\left(T_t^{\mathrm{out}}-T_t\right)+bN_t-cu_t$

The initial state is \(T_0=27\,^\circ\mathrm{C}\).

Changing \(u\) changes the real cooling action. The temperature path \(T_1,\ldots,T_n\) is a consequence of that action, so temperature is a system state rather than a decision variable.

The next cell implements this carried-forward model and the evaluation functions used throughout the lecture.


In [ ]:
# The classroom system and terminology are carried forward from Lecture 01-1.
import numpy as np

TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING = 0.0
MAX_COOLING = 5.0
MIN_TEMPERATURE = 20.0
MAX_TEMPERATURE = 30.0
MAX_ENERGY = 60.0


def expand_decision(decision):
    """Expand (early cooling, late cooling) into the 12-step decision schedule."""
    early_cooling, late_cooling = map(float, decision)
    return np.r_[
        np.full(6, early_cooling),
        np.full(6, late_cooling),
    ]


def simulate_classroom(decision):
    """Apply the 01-1 state equation and return the complete state path."""
    cooling_schedule = expand_decision(decision)
    temperatures = [INITIAL_TEMPERATURE]

    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        next_temperature = (
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )
        temperatures.append(next_temperature)

    return cooling_schedule, np.asarray(temperatures)


def performance_outputs(cooling_schedule, temperatures):
    """Calculate the raw performance outputs D and E defined in 01-1."""
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule ** 2)
    return float(discomfort), float(energy)


def evaluate_candidate(decision, energy_weight=1.0):
    """Simulate one decision, check requirements, and calculate its score."""
    decision = tuple(map(float, decision))
    cooling_schedule, temperatures = simulate_classroom(decision)
    discomfort, energy = performance_outputs(cooling_schedule, temperatures)

    violations = []
    if not all(MIN_COOLING <= value <= MAX_COOLING for value in decision):
        violations.append("cooling bound")
    if temperatures[1:].min() < MIN_TEMPERATURE:
        violations.append("minimum temperature")
    if temperatures[1:].max() > MAX_TEMPERATURE:
        violations.append("maximum temperature")
    if energy > MAX_ENERGY:
        violations.append("energy limit")

    return {
        "decision": decision,
        "cooling_schedule": cooling_schedule,
        "temperatures": temperatures,
        "discomfort": discomfort,
        "energy": energy,
        "energy_weight": float(energy_weight),
        "objective": discomfort + float(energy_weight) * energy,
        "feasible": not violations,
        "violations": tuple(violations),
    }


def enumerate_grid_candidates(energy_weight=1.0, step=0.5):
    """Evaluate the transparent decision grid used in Lectures 01-1 and 01-2."""
    levels = np.arange(MIN_COOLING, MAX_COOLING + step / 2, step)
    return [
        evaluate_candidate((early, late), energy_weight)
        for early in levels
        for late in levels
    ]


def best_grid_candidate(energy_weight=1.0, step=0.5):
    """Return the lowest-scoring feasible candidate on the stated grid."""
    candidates = enumerate_grid_candidates(energy_weight, step)
    return min(
        (candidate for candidate in candidates if candidate["feasible"]),
        key=lambda candidate: candidate["objective"],
    )


### 3 · Trace one candidate through the existing model

For the candidate \(u=(3,2)\), the first six cooling actions are 3 and the last six are 2. Evaluation follows the same causal order introduced in 01-1:

> **choose \(u\) → simulate \(T\) → calculate \(D,E\) → check constraints → calculate \(J\)**

The next cell compares three useful candidates:

- \((1,1)\) uses little energy but leaves the room uncomfortable.
- \((3,2)\) balances discomfort and energy under the default weight \(\lambda_E=1\).
- \((4,4)\) cools aggressively and violates requirements.

A candidate must pass every constraint before its objective score can make it eligible for selection.


In [ ]:
import sys
from types import SimpleNamespace

import matplotlib

def _pyplot(*, interactive=False):
    """Return pyplot, activating ipympl for interactive figures when available."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt

    return plt

def show_candidate_paths(
    evaluate_candidate,
    *,
    time_steps=12,
    min_temperature=20.0,
    max_temperature=30.0,
):
    """Print and plot the 01-2 candidate comparison."""
    plt = _pyplot()
    decisions = ((1.0, 1.0), (3.0, 2.0), (4.0, 4.0))
    results = [evaluate_candidate(decision, energy_weight=1.0) for decision in decisions]

    print(
        f"{'decision':>12} | {'min T':>6} | {'max T':>6} | "
        f"{'D':>8} | {'E':>6} | {'J':>8} | {'feasible':>8} | reason"
    )
    print("-" * 91)
    for result in results:
        reason = ", ".join(result["violations"]) or "—"
        print(
            f"{str(result['decision']):>12} | "
            f"{result['temperatures'][1:].min():>6.2f} | "
            f"{result['temperatures'][1:].max():>6.2f} | "
            f"{result['discomfort']:>8.2f} | {result['energy']:>6.2f} | "
            f"{result['objective']:>8.2f} | "
            f"{str(result['feasible']):>8} | {reason}"
        )

    figure, axis = plt.subplots(figsize=(8.8, 4.2))
    axis.axhspan(
        min_temperature,
        max_temperature,
        color="tab:blue",
        alpha=0.07,
        label="Allowed temperature range",
    )
    axis.axhspan(22, 24, color="tab:green", alpha=0.18, label="Comfort target")
    colors = ("tab:orange", "tab:blue", "tab:red")
    for result, color in zip(results, colors):
        early, late = result["decision"]
        status = "feasible" if result["feasible"] else "infeasible"
        axis.plot(
            np.arange(time_steps + 1),
            result["temperatures"],
            marker="o",
            linewidth=2,
            color=color,
            label=f"u=({early:g}, {late:g}) · {status}",
        )
    axis.set(
        xlabel="Time step",
        ylabel="Indoor temperature (°C)",
        title="A decision produces a state path; the path is not the decision",
        xlim=(0, time_steps),
        ylim=(18.5, 31),
    )
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8, ncol=2)
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    return results

candidate_results = show_candidate_paths(
    evaluate_candidate,
    time_steps=TIME_STEPS,
    min_temperature=MIN_TEMPERATURE,
    max_temperature=MAX_TEMPERATURE,
)

The low-cooling candidate \((1,1)\) is feasible because its temperature remains inside the wider 20–30 °C allowed range and its energy stays below 60. It still has high discomfort. **Feasible** means “allowed,” not “good” or “optimal.”

The candidate \((4,4)\) is rejected because it exceeds the energy limit and eventually falls below the minimum allowed temperature. Its score cannot rescue it.

### 4 · Separate raw performance from the objective

The state path and decision schedule produce two raw performance outputs:

> $\displaystyle D(u)=\sum_{t=1}^{n}\left[\max(T_t(u)-24,0)^2+\max(22-T_t(u),0)^2\right]$
>
> $\displaystyle E(u)=\frac{1}{2}\sum_{t=0}^{n-1}u_t^2,\qquad n=12.$

These outputs describe what happened. The scalar score combines them:

> $\displaystyle J(u;\lambda_E)=H\!\left(D(u),E(u);\lambda_E\right)=D(u)+\lambda_EE(u).$

The **objective** is to minimize \(J\) over feasible decisions. The energy weight \(\lambda_E\) is a hyperparameter: it changes how the same \((D,E)\) pair is valued, but it does not change the temperature path, raw performance, or feasibility of a fixed decision.

| Change | Temperature path | $D,E$ | Feasibility | Score and preferred decision |
|:---|:---:|:---:|:---:|:---:|
| Change decision $u$ | Can change | Can change | Can change | Can change |
| Change weight $\lambda_E$ only | Unchanged | Unchanged | Unchanged | Can change |


In [ ]:
def show_weight_comparison(enumerate_candidates, best_candidate):
    """Show how the energy weight changes the selected feasible grid candidate."""
    plt = _pyplot()
    energy_weights = (0.0, 1.0, 3.0)
    feasible = [
        candidate
        for candidate in enumerate_candidates(energy_weight=0.0)
        if candidate["feasible"]
    ]
    selected = [best_candidate(weight) for weight in energy_weights]

    print(f"{'lambda_E':>8} | {'best grid decision':>20} | {'D':>8} | {'E':>6} | {'J':>8}")
    print("-" * 65)
    for result in selected:
        print(
            f"{result['energy_weight']:>8.1f} | {str(result['decision']):>20} | "
            f"{result['discomfort']:>8.2f} | {result['energy']:>6.2f} | "
            f"{result['objective']:>8.2f}"
        )

    figure, axis = plt.subplots(figsize=(7.6, 4.8))
    axis.scatter(
        [candidate["energy"] for candidate in feasible],
        [candidate["discomfort"] for candidate in feasible],
        color="lightgray",
        edgecolor="white",
        s=60,
        label="Feasible grid candidate",
    )
    colors = ("tab:green", "tab:blue", "tab:purple")
    for result, color in zip(selected, colors):
        axis.scatter(
            result["energy"],
            result["discomfort"],
            marker="*",
            s=210,
            color=color,
            edgecolor="black",
            label=(
                f"best at $\\lambda_E$={result['energy_weight']:g}: "
                f"u=({result['decision'][0]:g}, {result['decision'][1]:g})"
            ),
            zorder=3,
        )
    axis.set(
        xlabel="Raw energy performance $E$",
        ylabel="Raw discomfort performance $D$",
        title="Same feasible candidates, different evaluation weights",
    )
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    return selected

best_by_weight = show_weight_comparison(
    enumerate_grid_candidates, best_grid_candidate
)

### 5 · State the constraints and assemble the classroom decision problem

Three types of requirement define the feasible set:

| Constraint type | Mathematical statement | Fixed value |
|:---|:---|:---|
| Decision bounds | $u_{\min}\le u_{\mathrm{early}},u_{\mathrm{late}}\le u_{\max}$ | $u_{\min}=0,\ u_{\max}=5$ |
| State bounds | $T_{\min}\le T_t\le T_{\max}$, $t=1,\ldots,n$ | $T_{\min}=20\,^\circ\mathrm{C},\ T_{\max}=30\,^\circ\mathrm{C}$ |
| Resource limit | $E(u)\le E_{\max}$ | $E_{\max}=60$ |

The 22–24 °C comfort range is deliberately part of the discomfort output \(D\), not a hard constraint. Temperatures outside that target can be allowed while receiving a worse objective score.

We can now assemble the concrete classroom decision problem. Lecture 02-1 will use this exact instance to introduce the general symbols \(x,y,f,g,h\).

> $\displaystyle \underset{u}{\operatorname{minimize}}\quad
J(u;\lambda_E)=D(u)+\lambda_EE(u)$
>
> $\displaystyle \text{using}\quad
u_t=\begin{cases}
u_{\mathrm{early}},&t=0,\ldots,5,\\
u_{\mathrm{late}},&t=6,\ldots,11,
\end{cases}$
>
> $\displaystyle \text{and}\quad
T_{t+1}=T_t+a(T_t^{\mathrm{out}}-T_t)+bN_t-cu_t,$
>
> $\displaystyle \text{subject to}\quad
u_{\min}\le u_{\mathrm{early}},u_{\mathrm{late}}\le u_{\max},\qquad
T_{\min}\le T_t\le T_{\max}\quad(t=1,\ldots,n),\qquad E(u)\le E_{\max},$
>
> with $\displaystyle n=12,\quad
(u_{\min},u_{\max})=(0,5),\quad
(T_{\min},T_{\max})=(20,30),\quad E_{\max}=60.$

The state equation predicts behavior. The requirement constraints decide whether that behavior is acceptable. Both appear in the formulation, but their jobs remain distinct.

### 6 · Search the stated candidate set

As in 01-1, we inspect cooling pairs on a 0.5-unit grid. This finite grid is the candidate set used by the demonstration. The star below is therefore the best **grid candidate**, not a claim about every continuous value between grid points.


In [ ]:
def show_objective_landscape(
    enumerate_candidates,
    best_candidate,
    *,
    min_cooling=0.0,
    max_cooling=5.0,
):
    """Plot feasible objective values on the stated 0.5-unit grid."""
    plt = _pyplot()
    step = 0.5
    records = enumerate_candidates(energy_weight=1.0, step=step)
    levels = np.arange(min_cooling, max_cooling + step / 2, step)
    values = np.full((len(levels), len(levels)), np.nan)
    for record in records:
        early, late = record["decision"]
        if record["feasible"]:
            values[
                int(round((early - min_cooling) / step)),
                int(round((late - min_cooling) / step)),
            ] = record["objective"]
    selected = best_candidate(energy_weight=1.0, step=step)
    infeasible = [record for record in records if not record["feasible"]]

    figure, axis = plt.subplots(figsize=(6.6, 5.2))
    color_map = plt.colormaps["viridis_r"].copy()
    color_map.set_bad("#e6e6e6")
    image = axis.imshow(
        values,
        origin="lower",
        cmap=color_map,
        extent=(min_cooling - 0.25, max_cooling + 0.25) * 2,
        aspect="equal",
    )
    axis.scatter(
        [record["decision"][1] for record in infeasible],
        [record["decision"][0] for record in infeasible],
        marker="x",
        color="#9c9c9c",
        s=34,
        label="Infeasible",
    )
    axis.scatter(
        selected["decision"][1],
        selected["decision"][0],
        marker="*",
        s=240,
        color="gold",
        edgecolor="black",
        label="Lowest-score feasible grid candidate",
        zorder=3,
    )
    axis.set(
        xticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        yticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        xlabel="Late cooling decision $u_{\\mathrm{late}}$",
        ylabel="Early cooling decision $u_{\\mathrm{early}}$",
        title="Feasible objective landscape at $\\lambda_E=1$",
    )
    axis.legend(fontsize=8, loc="upper right")
    figure.colorbar(image, ax=axis, label="Objective score $J$")
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    print(
        "Best 0.5-grid candidate at lambda_E=1: "
        f"u={selected['decision']}, D={selected['discomfort']:.2f}, "
        f"E={selected['energy']:.2f}, J={selected['objective']:.2f}"
    )
    return selected

best_at_weight_one = show_objective_landscape(
    enumerate_grid_candidates,
    best_grid_candidate,
    min_cooling=MIN_COOLING,
    max_cooling=MAX_COOLING,
)

### 7 · Explore one role at a time

The final figure keeps every definition above fixed and exposes three controls:

- The two **blue** sliders change the decision \(u=(u_{\mathrm{early}},u_{\mathrm{late}})\).
- The **purple** slider changes the energy-weight hyperparameter \(\lambda_E\).
- The square marks the current candidate; the star marks the lowest-scoring feasible 0.5-grid candidate.

First keep \(\lambda_E=1\) and change a cooling decision. Observe that the temperature path, \(D\), \(E\), feasibility, and \(J\) can all change. Then keep the decision fixed and change only \(\lambda_E\). The physical path, raw outputs, and feasibility stay fixed while the score and preferred grid candidate can change.


In [ ]:
def show_classroom_explorer(
    evaluate_candidate,
    enumerate_candidates,
    *,
    time_steps=12,
    min_cooling=0.0,
    max_cooling=5.0,
    min_temperature=20.0,
    max_temperature=30.0,
    initial_decision=(3.0, 2.0),
):
    """Build the shared decision/state/performance interactive explorer."""
    plt = _pyplot(interactive=True)
    from matplotlib.widgets import Slider

    plt.close("all")
    step = 0.5
    baseline = enumerate_candidates(energy_weight=1.0, step=step)
    levels = np.arange(min_cooling, max_cooling + step / 2, step)
    feasible_points = np.array([
        (record["energy"], record["discomfort"])
        for record in baseline if record["feasible"]
    ])
    infeasible_points = np.array([
        (record["energy"], record["discomfort"])
        for record in baseline if not record["feasible"]
    ])

    def objective_landscape(energy_weight):
        values = np.full((len(levels), len(levels)), np.nan)
        records = enumerate_candidates(energy_weight, step=step)
        for record in records:
            early, late = record["decision"]
            if record["feasible"]:
                values[
                    int(round((early - min_cooling) / step)),
                    int(round((late - min_cooling) / step)),
                ] = record["objective"]
        best = min(
            (record for record in records if record["feasible"]),
            key=lambda record: record["objective"],
        )
        return values, best

    initial_weight = 1.0
    current = evaluate_candidate(initial_decision, initial_weight)
    initial_landscape, best = objective_landscape(initial_weight)
    figure, axes = plt.subplots(1, 3, figsize=(12, 8))
    figure.subplots_adjust(left=0.07, right=0.98, bottom=0.35, top=0.80, wspace=0.34)

    axes[0].axhspan(
        min_temperature,
        max_temperature,
        color="tab:blue",
        alpha=0.07,
        label="Allowed range",
    )
    axes[0].axhspan(22, 24, color="tab:green", alpha=0.18, label="Comfort target")
    temperature_line, = axes[0].plot(
        np.arange(time_steps + 1),
        current["temperatures"],
        marker="o",
        linewidth=2,
        color="tab:blue",
    )
    axes[0].set(
        xlabel="Time step",
        ylabel="Indoor temperature (°C)",
        xlim=(0, time_steps),
        ylim=(17, 34),
    )
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)

    axes[1].scatter(
        infeasible_points[:, 0],
        infeasible_points[:, 1],
        marker="x",
        color="lightgray",
        label="Infeasible grid candidate",
    )
    axes[1].scatter(
        feasible_points[:, 0],
        feasible_points[:, 1],
        color="teal",
        alpha=0.55,
        label="Feasible grid candidate",
    )
    current_performance = axes[1].scatter(
        current["energy"], current["discomfort"],
        marker="s", s=90, color="tab:orange", edgecolor="black",
        label="Current candidate",
    )
    best_performance = axes[1].scatter(
        best["energy"], best["discomfort"],
        marker="*", s=190, color="gold", edgecolor="black",
        label="Best grid candidate",
    )
    all_points = feasible_points.tolist() + infeasible_points.tolist()
    axes[1].set(
        xlabel="Raw energy performance $E$",
        ylabel="Raw discomfort performance $D$",
        xlim=(0, max(point[0] for point in all_points) * 1.07),
        ylim=(0, max(point[1] for point in all_points) * 1.07),
        title="Raw performance and feasibility",
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=7)

    color_map = plt.colormaps["viridis_r"].copy()
    color_map.set_bad("#e6e6e6")
    objective_image = axes[2].imshow(
        initial_landscape,
        origin="lower",
        cmap=color_map,
        extent=(min_cooling - 0.25, max_cooling + 0.25) * 2,
        aspect="equal",
    )
    current_decision = axes[2].scatter(
        initial_decision[1], initial_decision[0],
        marker="s", s=90, color="tab:orange", edgecolor="black",
        label="Current candidate",
    )
    best_decision = axes[2].scatter(
        best["decision"][1], best["decision"][0],
        marker="*", s=190, color="gold", edgecolor="black",
        label="Best grid candidate",
    )
    axes[2].set(
        xticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        yticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        xlabel="Late cooling decision",
        ylabel="Early cooling decision",
    )
    axes[2].legend(fontsize=8)

    status_text = figure.text(
        0.5, 0.955, "", ha="center", va="top", fontsize=12, fontweight="bold"
    )
    metrics_text = figure.text(0.5, 0.915, "", ha="center", va="top", fontsize=10)
    figure.text(
        0.5,
        0.255,
        "Fixed system model: a=0.12, b=0.012, c=0.45; "
        "external inputs: outdoor temperature=31 °C, occupants=20",
        ha="center",
        va="center",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#f1f3f5", edgecolor="#adb5bd"),
    )

    early_axis = figure.add_axes([0.35, 0.175, 0.55, 0.025])
    late_axis = figure.add_axes([0.35, 0.115, 0.55, 0.025])
    weight_axis = figure.add_axes([0.35, 0.055, 0.55, 0.025])
    early_slider = Slider(
        early_axis,
        "Decision · early cooling",
        min_cooling,
        max_cooling,
        valinit=initial_decision[0],
        valstep=0.25,
        valfmt="%1.2f",
        color="tab:blue",
    )
    late_slider = Slider(
        late_axis,
        "Decision · late cooling",
        min_cooling,
        max_cooling,
        valinit=initial_decision[1],
        valstep=0.25,
        valfmt="%1.2f",
        color="tab:blue",
    )
    weight_slider = Slider(
        weight_axis,
        "Hyperparameter · $\\lambda_E$",
        0.0,
        3.0,
        valinit=initial_weight,
        valstep=0.1,
        valfmt="%1.1f",
        color="tab:purple",
    )
    state = {}

    def refresh(_=None):
        decision = (early_slider.val, late_slider.val)
        energy_weight = weight_slider.val
        current = evaluate_candidate(decision, energy_weight)
        landscape, best = objective_landscape(energy_weight)
        status = "FEASIBLE" if current["feasible"] else "INFEASIBLE"
        temperature_line.set_ydata(current["temperatures"])
        current_performance.set_offsets([[current["energy"], current["discomfort"]]])
        best_performance.set_offsets([[best["energy"], best["discomfort"]]])
        objective_image.set_data(landscape)
        objective_image.set_clim(np.nanmin(landscape), np.nanmax(landscape))
        current_decision.set_offsets([[decision[1], decision[0]]])
        best_decision.set_offsets([[best["decision"][1], best["decision"][0]]])
        axes[0].set_title(f"State path: {status.lower()}")
        axes[2].set_title(f"Objective landscape\n$\\lambda_E$={energy_weight:.1f}")
        status_text.set_text(
            f"Current decision u=({decision[0]:.2f}, {decision[1]:.2f}) · {status}"
        )
        status_text.set_color("#087f5b" if current["feasible"] else "#c92a2a")
        eligibility = "eligible" if current["feasible"] else "rejected before comparison"
        metrics_text.set_text(
            f"D={current['discomfort']:.2f}, E={current['energy']:.2f}, "
            f"J=D+{energy_weight:.1f}E={current['objective']:.2f} · {eligibility}  |  "
            f"best grid u=({best['decision'][0]:.1f}, {best['decision'][1]:.1f})"
        )
        state.clear()
        state.update(current=current, best=best, energy_weight=energy_weight)
        figure.canvas.draw_idle()

    for slider in (early_slider, late_slider, weight_slider):
        slider.on_changed(refresh)
    figure._classroom_sliders = (early_slider, late_slider, weight_slider)
    refresh()
    plt.show()
    return SimpleNamespace(
        figure=figure,
        state=state,
        refresh=refresh,
        early_slider=early_slider,
        late_slider=late_slider,
        energy_weight_slider=weight_slider,
    )

formulation_explorer = show_classroom_explorer(
    evaluate_candidate,
    enumerate_grid_candidates,
    time_steps=TIME_STEPS,
    min_cooling=MIN_COOLING,
    max_cooling=MAX_COOLING,
    min_temperature=MIN_TEMPERATURE,
    max_temperature=MAX_TEMPERATURE,
    initial_decision=(3.0, 2.0),
)
formulation_state = formulation_explorer.state

### Takeaway

Lecture 01-1 built the system model that maps a cooling decision to a temperature path and raw performance. Lecture 01-2 kept that model fixed and organized the ingredients of its decision problem:

> **name the decision → simulate the state → measure raw performance → check constraints → compare the objective → select a feasible candidate**

For any new problem, keep the same distinctions:

- A **decision variable** is directly chosen; a system state is produced by the system model.
- A **performance output** records what happened; an objective states how candidates are compared.
- A **constraint** determines eligibility; a good score cannot make an infeasible candidate acceptable.
- A **hyperparameter** changes the evaluation or search, not the physical outcome of a fixed decision.

Lecture 02-1 will translate this same reasoning into the standard vector-and-function form \(x\rightarrow y=\operatorname{Sim}(x)\rightarrow f,g,h\) without changing the meaning of any term.
